# Topic 8 practice: groupby

The tables are built in code, so no CSV is needed. Re-run the setup cell to reset them. Start each question from `orders` or `team`, and give each result a new name.

Tip: a comment line must start with `#`.

In [1]:
import pandas as pd

orders = pd.DataFrame({"order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
                       "customer": ["asha", "ravi", "zoya", "meena", "asha", "zoya", "ravi", "kiran"],
                       "city": ["Indore", "Delhi", "Pune", "Indore", "Indore", "Pune", "Delhi", "Pune"],
                       "category": ["Food", "Toys", "Books", "Food", "Books", "Food", "Toys", "Toys"],
                       "amount": [1200, 800, 1500, 950, 400, 300, 600, 1100]})

team = pd.DataFrame({"name": ["Anu", "Bala", "Chitra", "Dev", "Esha", "Farid"],
                     "dept": ["IT", "IT", "HR", "HR", "IT", "HR"],
                     "salary": [50000, None, 40000, 46000, 70000, None]})
print(orders)
print(team)

   order_id customer    city category  amount
0      1001     asha  Indore     Food    1200
1      1002     ravi   Delhi     Toys     800
2      1003     zoya    Pune    Books    1500
3      1004    meena  Indore     Food     950
4      1005     asha  Indore    Books     400
5      1006     zoya    Pune     Food     300
6      1007     ravi   Delhi     Toys     600
7      1008    kiran    Pune     Toys    1100
     name dept   salary
0     Anu   IT  50000.0
1    Bala   IT      NaN
2  Chitra   HR  40000.0
3     Dev   HR  46000.0
4    Esha   IT  70000.0
5   Farid   HR      NaN


### Q1 (easy). One number per group

Find the total `amount` for each `category` and save it in `c1`. Print `c1`. Then print the name of the category with the biggest total.

In [7]:
# your code here (add a comment saying why for each line)
o = orders.copy()
c1 = o.groupby("category")["amount"].sum()
print(c1) 
s = c1.reset_index()
print(c1.idxmax())   
print(s["amount"].max())

category
Books    1900
Food     2450
Toys     2500
Name: amount, dtype: int64
Toys
2500


### Q2 (easy). Several numbers per group

For each `customer` make a table with two columns: `bills` (how many orders) and `total` (sum of amount). Make `customer` a normal column. Save it in `c2` and print it.

In [25]:
# your code here (add a comment saying why for each line)
c2 = orders.groupby("customer").agg(
    bills=("order_id","count"),
    total=("amount","sum")
).reset_index()
print(c2)

  customer  bills  total
0     asha      2   1600
1    kiran      1   1100
2    meena      1    950
3     ravi      2   1400
4     zoya      2   1800


### Q3 (easy to medium). Average, rounded and sorted

Find the average `amount` per `city`, rounded to 1 decimal, sorted from the highest average to the lowest. Save in `c3`, print it and then print the list of city names in that order.

In [14]:
# your code here (add a comment saying why for each line)
c3 = orders.groupby("city")["amount"].mean().round(1).sort_values(ascending=False)
print(c3)

print(c3.index.tolist())

city
Pune      966.7
Indore    850.0
Delhi     700.0
Name: amount, dtype: float64
['Pune', 'Indore', 'Delhi']


### Q4 (medium). Two group columns

Group by `city` and `category` together. Make a normal table with `total` (sum of amount) and `bills` (count of order_id). Save in `c4`, print it, and print how many rows it has.

In [ ]:
# your code here (add a comment saying why for each line)
c4 = orders.groupby(["city", "category"], as_index=False).agg(
    total=("amount", "sum"),    # sum of amount in each pile
    bills=("order_id", "count"),   # how many orders in each pile
)
print(c4)
print(len(c4))


     city category  total  bills
0   Delhi     Toys   1400      2
1  Indore    Books    400      1
2  Indore     Food   2150      2
3    Pune    Books   1500      1
4    Pune     Food    300      1
5    Pune     Toys   1100      1


### Q5 (medium). Keep only some groups

Keep the original rows of customers whose total spending (all their orders added up) is above 1500. Save in `c5`, print it, and print the list of `order_id` values in `c5`.

In [26]:
# your code here (add a comment saying why for each line)
# keep a customer's rows only if that customer's total is above 1500
c5 = orders.groupby("customer").filter(lambda g: g["amount"].sum() > 1500)
print(c5)
print(c5["order_id"].tolist())      # [1001, 1003, 1005, 1006] -> asha (1600) and zoya (1800)

# the same in 2 steps
t = orders.groupby("customer")["amount"].sum()      # total per customer
big = t[t > 1500].index                             # customers who passed
print(orders[orders["customer"].isin(big)]["order_id"].tolist())   # same list

   order_id customer    city category  amount
0      1001     asha  Indore     Food    1200
2      1003     zoya    Pune    Books    1500
4      1005     asha  Indore    Books     400
5      1006     zoya    Pune     Food     300
[1001, 1003, 1005, 1006]
[1001, 1003, 1005, 1006]


### Q6 (medium to hard). Share and rank inside a group

On a copy of `orders` called `o`: add `city_total` (the total of that row's city), then `share_pct` (the row's amount as a percent of its city total, rounded to 1 decimal). Print the sum of `share_pct` for each city, rounded to a whole number, to check it is about 100. Then add `rank_in_city` (biggest amount = 1, whole numbers) and print the `order_id` values that are rank 1 or 2.

In [27]:
# your code here (add a comment saying why for each line)
o = orders.copy()                                                      # copy, because we add columns
o["city_total"] = o.groupby("city")["amount"].transform("sum")         # city total on every row
o["share_pct"] = (o["amount"] / o["city_total"] * 100).round(1)        # slice / whole * 100
print(o.groupby("city")["share_pct"].sum().round(0))                   # each city adds up to about 100
o["rank_in_city"] = o.groupby("city")["amount"].rank(ascending=False).astype(int)   # biggest = 1, restarts in every city
print(o[o["rank_in_city"] <= 2]["order_id"].tolist())                  # rank 1 or 2

city
Delhi     100.0
Indore    100.0
Pune      100.0
Name: share_pct, dtype: float64
[1001, 1002, 1003, 1004, 1007, 1008]


### Q7 (hard). Fill gaps by group

In `team`, two salaries are missing. On a copy called `t`, fill each missing salary with the average salary of that person's `dept`, into a new column `filled`. Print `t`. Then add `rank_in_dept` (highest filled salary = 1, whole numbers) and print the names of the people with rank 1.

In [29]:
t = team.copy()                                                         # copy, because we add columns
t["filled"] = t["salary"].fillna(t.groupby("dept")["salary"].transform("mean"))   # each gap gets its own dept's average
print(t)
t["rank_in_dept"] = t.groupby("dept")["filled"].rank(ascending=False).astype(int)  # highest salary = 1 in each dept
print(t[t["rank_in_dept"] == 1]["name"].tolist())                       # the top earner of each dept

     name dept   salary   filled
0     Anu   IT  50000.0  50000.0
1    Bala   IT      NaN  60000.0
2  Chitra   HR  40000.0  40000.0
3     Dev   HR  46000.0  46000.0
4    Esha   IT  70000.0  70000.0
5   Farid   HR      NaN  43000.0
['Dev', 'Esha']
